# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` and required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset overview
print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.datePublished}\nIdentifier: {metadata.identifier}\nVersion: {metadata.version}\nLicense: {metadata.license}\n")
print(f"Keywords: {', '.join(metadata.keywords)}")


## 2. Data Overview
Review available record sets, fields, and their IDs (referenced by their `@id`).


In [ ]:
# List all record sets in the dataset
print("Available Record Sets (@id and name):")
for rs in metadata.record_sets:
    print(f"- @id: {rs['@id']}, name: {rs['name']}")


In [ ]:
# Examine fields and columns for each record set
print("\nFields in each record set:")
for rs in metadata.record_sets:
    print(f"\nRecord Set @id: {rs['@id']} ({rs['name']}):")
    if 'fields' in rs:
        for field in rs['fields']:
            print(f"  - Field @id: {field['@id']}, name: {field.get('name')} (dataType: {field.get('dataType')})")
    if 'columns' in rs:
        for col in rs['columns']:
            print(f"  - Column @id: {col['@id']} (column name: {col.get('name')})")


## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis, referencing entities by their `@id`.


In [ ]:
# Choose record set @ids to extract
record_sets_ids = [rs['@id'] for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame with shape: {dataframes[record_set_id].shape}\nColumns: {dataframes[record_set_id].columns.tolist()}\n")
    else:
        print("  (No records found for this record set.)\n")
# Use one record set for demonstration below
if dataframes:
    chosen_record_set_id = list(dataframes.keys())[0]
    print(f"Sample of {chosen_record_set_id} records:")
    display(dataframes[chosen_record_set_id].head())
else:
    chosen_record_set_id = None
    print("No record sets with records were found.")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping data. All field/column references use their `@id`.


In [ ]:
if chosen_record_set_id:
    df = dataframes[chosen_record_set_id]
    print(f"Extracted DataFrame: {chosen_record_set_id} ({df.shape[0]} rows)")

    # Select a numeric field by @id for demonstration
    # Let's detect numeric fields (float/integer) and pick one
    num_field_id = None
    # Get list of fields for this record set
    fields = None
    for rs in metadata.record_sets:
        if rs['@id'] == chosen_record_set_id:
            fields = rs.get('fields', [])
            break
    # Try to select the first float/integer field if present
    if fields:
        for field in fields:
            dt = field.get('dataType', '').lower()
            if 'float' in dt or 'number' in dt or 'int' in dt:
                num_field_id = field['@id']
                break

    if num_field_id is None:
        print("No numeric field detected in record set for analysis.")
    else:
        print(f"Using numeric field for analysis: {num_field_id}")
        # Filter based on a threshold (for demonstration threshold=10)
        threshold = 10
        filtered_df = df[df[num_field_id] > threshold]
        print(f"Filtered records with {num_field_id} > {threshold} (n={filtered_df.shape[0]}):")
        display(filtered_df.head())
        # Normalize numeric field
        filtered_df[f"{num_field_id}_normalized"] = (filtered_df[num_field_id] - filtered_df[num_field_id].mean()) / filtered_df[num_field_id].std()
        print(f"Normalized {num_field_id} for filtered records:")
        display(filtered_df[[num_field_id, f"{num_field_id}_normalized"]].head())
        # Attempt to group by a categorical field if present
        group_field_id = None
        for field in fields:
            dt = field.get('dataType', '').lower()
            if 'string' in dt or 'category' in dt or 'text' in dt:
                group_field_id = field['@id']
                break

        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[num_field_id].mean().reset_index()
            print(f"Grouped mean of {num_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No categorical field found for grouping.")
else:
    print("No DataFrame available for EDA.")


## 5. Visualization
Visualize data distribution for the selected numeric field (referenced by field's `@id`) and relation to a categorical field if detected.


In [ ]:
if chosen_record_set_id and num_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[num_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {num_field_id}')
    plt.xlabel(num_field_id)
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(12, 5))
        sns.boxplot(x=group_field_id, y=num_field_id, data=df)
        plt.title(f'{num_field_id} by {group_field_id}')
        plt.xticks(rotation=30)
        plt.show()
else:
    print("No numeric field to visualize.")


## 6. Conclusion

In this notebook, we loaded and explored the FAIR^2 mass spectrometry dataset via its Croissant schema. We:

- Loaded and reviewed metadata and available record sets via their `@id`
- Enumerated fields and columns for each record set
- Extracted tabular data and performed simple filtering, normalization, and grouping by key fields, always referencing by `@id`
- Visualized numeric field distributions

These steps can be adapted and expanded for deeper biological and computational investigations, or to integrate more advanced analysis and ML workflows with FAIR metadata compliance.
